In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from model.resnet11 import MnistFM

## Config

In [ ]:
## MNIST Data
mnist_data_dir = "/workspace/data"

## Train
exp_no = 1 # 実験番号
batch_size = 256
num_epochs = 1
checkpoint_interval = 10
save_dir = f"/workspace/checkpoints/exp{exp_no}"
log_csv_path = os.path.join(save_dir, "train_log.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## MNIST Data Check

In [ ]:
os.makedirs(mnist_data_dir, exist_ok=True)
train_dataset = torchvision.datasets.MNIST(root=mnist_data_dir, train=True, transform=transforms.ToTensor(), download=True)
test_dataset = torchvision.datasets.MNIST(root=mnist_data_dir, train=False, transform=transforms.ToTensor(), download=True)
print('dataset type: ', type(train_dataset))

In [ ]:
print('data length: ', len(train_dataset))

img, label = train_dataset[0]
print('img type: ', type(img))
print('img data type: ', img.dtype)
print('img size: ', img.shape)
print('label: ', label)
print('label type: ', type(label))

plt.imshow(img.view(-1, 28), cmap='gray')

## DataLoader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

# ==== data check ====
img, label = train_dataset[0]
print('train length: ', len(train_dataset))
print('train data sample size: ', img.shape)

labels = [label for _, label in train_dataset]
unique_labels = set(labels)
print('unique labels: ', unique_labels)
print('unique label length: ', len(unique_labels))

img, label = test_dataset[0]
print('test length: ', len(test_dataset))
print('test data sample size: ', img.shape)


## Train

In [ ]:
import os
import csv

os.makedirs(save_dir, exist_ok=True)

if not os.path.exists(log_csv_path):
    with open(log_csv_path, mode="w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "loss", "lr"])

model = MnistFM()
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for img, pseudo_prompt in train_loader:
        img = img.to(device) # (batch, 1, 28, 28)
        pseudo_prompt = pseudo_prompt.to(device) # (batch,)
        
        batch_size = pseudo_prompt.size(0)

        time_step = model.sample_time(batch_size, device) # (batch,)
        
        t = time_step[:, None, None, None] # (batch, 1, 1, 1)
        noise = model.sample_noise(img.shape, device) # (batch, 1, 28, 28)
        x_t = t * img + (1 - t) * noise # (batch, 1, 28, 28). time_step=tにおける中間ノイズの状態

        v_t = model(pseudo_prompt, time_step, x_t) # time_step=tにおける予測ベクトル場: v_t (batch, 1, 28, 28)
        u_t = img - noise # 正解ベクトル場: u_t (batch, 1, 28, 28)

        loss = F.mse_loss(v_t, u_t) # Flow Matching 論文内における損失関数の再現

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"epoch={epoch+1}, loss={avg_loss:.6f}")

    # 各epochのloss/lrをCSVに保存
    with open(log_csv_path, mode="a", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow([epoch + 1, avg_loss, current_lr])

    # 100epoch毎にcheckpoint保存
    if (epoch + 1) % checkpoint_interval == 0:
        checkpoint_path = os.path.join(save_dir, f"resnet11_epoch_{epoch+1}.pth")

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
        }, checkpoint_path)
        print(f"saved: {checkpoint_path}")

## Loss Visualization

In [ ]:
import pandas as pd

df = pd.read_csv(log_csv_path)

plt.figure(figsize=(8, 5))
plt.plot(df["epoch"], df["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()